# Canonical Correlation Analysis of 16S V1-V3 and V4 and metabolomics table with top VIPs

Date created: 1/29/2024

Notebook author: Yang Chen

Data analysis by: Britta de Pessemier, Yang Chen

In [97]:
import biom
from biom import load_table
from biom.util import biom_open
import numpy as np
import pandas as pd
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from skbio.stats.composition import clr
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

In [98]:
# Function to load BIOM table, sort rows by sum
def biom_to_df(biom_path):

    # Load BIOM table and convert to a DataFrame
    table = biom.load_table(biom_path)

    df = pd.DataFrame(table.matrix_data.toarray(),
                      index=table.ids(axis='observation'),
                      columns=table.ids(axis='sample'))
    
    
    # Sort rows by row sum in descending order
    df['row_sum'] = df.sum(axis=1)
    df = df.sort_values(by='row_sum', ascending=False)
    
    # Drop the 'row_sum' column before proceeding
    df = df.drop(columns=['row_sum'])

    # Transpose df
    df = df.T
    
    return df


In [99]:
def save_as_biom(df: pd.DataFrame, output_path: str):
    """
    Save a pandas DataFrame as a BIOM table.

    Parameters:
    df (pd.DataFrame): The DataFrame to save.
    output_path (str): Path to the output BIOM file.
    """
    table = biom.table.Table(df.values, observation_ids=df.index, sample_ids=df.columns)
    with biom_open(output_path, 'w') as f:
        table.to_hdf5(f, "save biom")

In [100]:
# Read in the metadata
metadata = pd.read_csv('../Metadata/metadata_final_18062024.tsv', sep='\t')
metadata

,SampleID,c_zone,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_face,zone,sample_type,planned_study_day_of_visit,visual_assessment_in_vivo_number_of_inflammatory_lesions_face,day,subject_randomization_number,visual_assessment_in_vivo_number_of_non_inflammatory_lesions_cheek_right,...,inflammatory_lesions_face,noninflammatory_lesions_face,a,cohort,subject_randomization_id,class,subject_ID,subject_ID_CC,zone_CC,group
0,LAMI.RD308.D16.C1,C1,not applicable,Lesional,skin,Day 16,not applicable,16,308,not applicable,...,0,0,33.765638,acne,RD308,acne,PP_308,PP_308C1,Lesional_C1,Acne_L
1,LAMI.RD310.D21.C1,C1,72,Lesional,skin,Day 21,36,21,310,17,...,36,72,31.919478,acne,RD310,acne,PP_310,PP_310C1,Lesional_C1,Acne_L
2,LAMI.RD305.D21.C3,C3,69,Non-lesional,skin,Day 21,26,21,305,25,...,26,69,22.152503,acne,RD305,healthy,PP_305,PP_305C3,Non-lesional_C3,Acne_NL
3,LAMI.RD306.D18.C2,C2,not applicable,Lesional,skin,Day 18,not applicable,18,306,not applicable,...,0,0,27.397918,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
4,LAMI.RD306.D7.C2,C2,90,Lesional,skin,Day 7,13,7,306,23,...,13,90,28.819451,acne,RD306,acne,PP_306,PP_306C2,Lesional_C2,Acne_L
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261,LAMI.RD317.D21.C1,C1,77,Lesional,skin,Day 21,19,21,317,20,...,19,77,21.946648,acne,RD317,acne,PP_317,PP_317C1,Lesional_C1,Acne_L
262,LAMI.RD001.D0.C1,C1,not applicable,Non-lesional,skin,Day 0,not applicable,0,1,not applicable,...,0,0,26.344240,control,RD001,healthy,PP_1,PP_1C1,Non-lesional_C1,Healthy
263,LAMI.RD014.D14.C2,C2,not applicable,Non-lesional,skin,Day 14,not applicable,14,14,not applicable,...,0,0,16.359699,control,RD014,healthy,PP_14,PP_14C2,Non-lesional_C2,Healthy
264,LAMI.RD314.D0.C1,C1,55,Lesional,skin,Day 0,31,0,314,16,...,31,55,22.494605,acne,RD314,acne,PP_314,PP_314C1,Lesional_C1,Acne_L


In [101]:
# Read in metabolomics table with top VIPs
metaB_tbl = pd.read_csv('../Data/metabolomics/Run3_10252024/output/metaB_top-VIPs_final.csv')
metaB_tbl = metaB_tbl.set_index('SampleID')

# Read in 16S V1-V3 table
V1V3_biom_path = '../Data/16S/Tables/16S_V1-V3_Genus_collapsed_absolute.biom'
V1V3_tbl = biom_to_df(V1V3_biom_path)

# Read in 16S V4 table
V4_biom_path = '../Data/16S/Tables/16S_V4_Genus_collapsed_absolute.biom'
V4_tbl = biom_to_df(V4_biom_path)

In [102]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_V1V3_tbl = metaB_tbl[metaB_tbl.index.isin(V1V3_tbl.index)]

# Remove the 'group' column (only for VIP table)
# metaB_V1V3_tbl = metaB_V1V3_tbl.drop(columns=['group'])
metaB_V1V3_tbl.index.name = None
metaB_V1V3_tbl

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.000,611709.700
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.690,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.920,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.000,427202.060
LAMI.RD304.D0.C2,24984.3220,2486485.20,1051277.400,459814.620,13703.5760,62253.426,223301.700
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.000,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.000,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.000,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.000,55986.290


In [103]:
# Convert to relative abundance (row-wise normalization)
metaB_V1V3_tbl_relative = metaB_V1V3_tbl.div(metaB_V1V3_tbl.sum(axis=1), axis=0)
metaB_V1V3_tbl_relative

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,0.009456,0.480551,0.230237,0.134635,0.000000,0.000000,0.145121
LAMI.RD304.D11.C1,0.014256,0.556292,0.231034,0.127297,0.002085,0.038587,0.030449
LAMI.RD304.D11.C2,0.000000,0.445655,0.191746,0.096169,0.001698,0.099395,0.165337
LAMI.RD304.D0.C1,0.002036,0.473830,0.178553,0.083500,0.002088,0.000000,0.259992
LAMI.RD304.D0.C2,0.005781,0.575333,0.243249,0.106394,0.003171,0.014404,0.051668
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.003977,0.508019,0.235307,0.202731,0.000000,0.000000,0.049966
LAMI.RD308.D9.C3,0.002365,0.528441,0.216669,0.202058,0.000000,0.000000,0.050467
LAMI.RD319.D2.C2,0.000000,0.544406,0.258387,0.123160,0.000000,0.000000,0.074047
LAMI.RD319.D2.C3,0.005702,0.433072,0.239277,0.095142,0.000000,0.000000,0.226807


In [104]:
# Save as biom for mmvec
metaB_V1V3_tbl_transposed = metaB_V1V3_tbl_relative.T
output_path = '../Data/multi-omics/metabolomics-tbl_16S_V1V3-matched_relative.biom'
save_as_biom(metaB_V1V3_tbl_transposed, output_path)

In [105]:
# Subset V1V3_tbl rows where sample is in metaB_tbl
V1V3_tbl_matched = V1V3_tbl[V1V3_tbl.index.isin(metaB_V1V3_tbl.index)]

top_V1V3_features = [' g__Cutibacterium', ' g__Staphylococcus',
                     ' g__Streptococcus', ' g__Corynebacterium', ' g__Lawsonella',
                     ' g__Micrococcus', ' g__Alloprevotella', ' g__Lactobacillus', 
                     ' g__Rothia', ' g__Kocuria', ' g__Haemophilus']

# Find the intersection of desired columns and actual DataFrame columns
available_columns = V1V3_tbl_matched.columns.intersection(top_V1V3_features)
V1V3_tbl_matched = V1V3_tbl_matched[available_columns]

V1V3_tbl_matched

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,2362.0,94.0,292.0,175.0,91.0,37.0,18.0,1.0,50.0,87.0,0.0
LAMI.RD001.D14.C1,2234.0,83.0,240.0,492.0,253.0,30.0,30.0,60.0,12.0,8.0,0.0
LAMI.RD001.D14.C2,1761.0,137.0,446.0,456.0,151.0,16.0,100.0,26.0,31.0,58.0,6.0
LAMI.RD001.D28.C1,2367.0,118.0,293.0,365.0,217.0,22.0,38.0,17.0,14.0,10.0,5.0
LAMI.RD002.D0.C2,2900.0,373.0,159.0,39.0,13.0,61.0,0.0,1.0,27.0,11.0,54.0
...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D7.C3,981.0,21.0,0.0,2.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD318.D28.C3,3218.0,5.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C1,1900.0,8.0,0.0,6.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMI.RD319.D14.C2,2315.0,8.0,0.0,9.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0


In [106]:
# Convert to relative abundance (row-wise normalization)
V1V3_tbl_matched_relative = V1V3_tbl_matched.div(V1V3_tbl_matched.sum(axis=1), axis=0)
V1V3_tbl_matched_relative

,g__Cutibacterium,g__Staphylococcus,g__Streptococcus,g__Corynebacterium,g__Lawsonella,g__Micrococcus,g__Alloprevotella,g__Lactobacillus,g__Rothia,g__Haemophilus,g__Kocuria
LAMI.RD001.D0.C1,0.736514,0.029311,0.091051,0.054568,0.028375,0.011537,0.005613,0.000312,0.015591,0.027128,0.000000
LAMI.RD001.D14.C1,0.649041,0.024114,0.069727,0.142940,0.073504,0.008716,0.008716,0.017432,0.003486,0.002324,0.000000
LAMI.RD001.D14.C2,0.552384,0.042974,0.139900,0.143036,0.047365,0.005019,0.031368,0.008156,0.009724,0.018193,0.001882
LAMI.RD001.D28.C1,0.682920,0.034045,0.084535,0.105309,0.062608,0.006347,0.010964,0.004905,0.004039,0.002885,0.001443
LAMI.RD002.D0.C2,0.797141,0.102529,0.043705,0.010720,0.003573,0.016767,0.000000,0.000275,0.007422,0.003024,0.014843
...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D7.C3,0.972250,0.020813,0.000000,0.001982,0.004955,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD318.D28.C3,0.995976,0.001548,0.000000,0.000000,0.002476,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD319.D14.C1,0.988554,0.004162,0.000000,0.003122,0.004162,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
LAMI.RD319.D14.C2,0.981348,0.003391,0.000000,0.003815,0.011446,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [107]:
# Save metadata for V1V3
metadata_V1V3 = metadata[metadata['SampleID'].isin(V1V3_tbl_matched_relative.index)]
metadata_V1V3 = metadata_V1V3[['SampleID', 'group']]

# Rename the first column to 'sample id'
metadata_V1V3.rename(columns={metadata_V1V3.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V1V3.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V1V3.columns[1:])]

metadata_V1V3.to_csv('../Data/multi-omics/16S_V1V3_metaB-matched.csv', index=0)
metadata_V1V3

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
7,LAMI.RD305.D23.C1,Acne_L
...,...,...
260,LAMI.RD305.D0.C2,Acne_L
261,LAMI.RD317.D21.C1,Acne_L
262,LAMI.RD001.D0.C1,Healthy
263,LAMI.RD014.D14.C2,Healthy


In [108]:
# Save as biom for mmvec
V1V3_tbl_matched_transposed = V1V3_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V1V3-tbl_metaB-matched_relative.biom'
save_as_biom(V1V3_tbl_matched_transposed, output_path)

In [109]:
# Create column_names_df with 'feature id' as the column header
column_names_df = pd.DataFrame({"feature id": metaB_V1V3_tbl.columns})  # Ensure feature id is a string

column_names_df

,feature id
0,Urocanic acid
1,Isoleucine
2,Phenylalanine
3,Tryptophan
4,N-acyl lipid glutamine-C14:0
5,Sorbitane Monooleate
6,C19H36O4 Fatty alcohol


In [110]:
# Save to a CSV file
output_path = "../Data/multi-omics/metabolite_info.csv"
column_names_df = column_names_df.to_csv(output_path, index=0)

In [111]:
# Subset metaB rows where sample is in V4_tbl
metaB_V4_tbl = metaB_tbl[metaB_tbl.index.isin(V4_tbl.index)]
# metaB_V4_tbl = metaB_V4_tbl.drop('group', axis=1)
metaB_V4_tbl.index.name = None
metaB_V4_tbl

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,39859.7580,2025607.40,970487.500,567510.060,0.0000,0.000,611709.700
LAMI.RD304.D11.C1,36750.4000,1434033.20,595568.940,328151.120,5375.8525,99470.690,78491.530
LAMI.RD304.D11.C2,0.0000,706197.75,303845.300,152392.720,2691.0256,157503.920,261996.800
LAMI.RD304.D0.C1,3345.8184,778566.10,293386.340,137201.390,3431.0034,0.000,427202.060
LAMI.RD304.D0.C2,24984.3220,2486485.20,1051277.400,459814.620,13703.5760,62253.426,223301.700
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,9787.1210,1250164.60,579058.800,498893.780,0.0000,0.000,122958.260
LAMI.RD308.D9.C3,3076.9688,687527.06,281896.700,262886.470,0.0000,0.000,65659.700
LAMI.RD319.D2.C2,0.0000,538412.94,255542.720,121803.890,0.0000,0.000,73231.560
LAMI.RD319.D2.C3,1407.5230,106901.95,59064.336,23485.404,0.0000,0.000,55986.290


In [112]:
# Convert to relative abundance (row-wise normalization)
metaB_V4_tbl_relative = metaB_V4_tbl.div(metaB_V4_tbl.sum(axis=1), axis=0)
metaB_V4_tbl_relative

,Urocanic acid,Isoleucine,Phenylalanine,Tryptophan,N-acyl lipid glutamine-C14:0,Sorbitane Monooleate,C19H36O4 Fatty alcohol
LAMI.RD001.D0.C1,0.009456,0.480551,0.230237,0.134635,0.000000,0.000000,0.145121
LAMI.RD304.D11.C1,0.014256,0.556292,0.231034,0.127297,0.002085,0.038587,0.030449
LAMI.RD304.D11.C2,0.000000,0.445655,0.191746,0.096169,0.001698,0.099395,0.165337
LAMI.RD304.D0.C1,0.002036,0.473830,0.178553,0.083500,0.002088,0.000000,0.259992
LAMI.RD304.D0.C2,0.005781,0.575333,0.243249,0.106394,0.003171,0.014404,0.051668
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.003977,0.508019,0.235307,0.202731,0.000000,0.000000,0.049966
LAMI.RD308.D9.C3,0.002365,0.528441,0.216669,0.202058,0.000000,0.000000,0.050467
LAMI.RD319.D2.C2,0.000000,0.544406,0.258387,0.123160,0.000000,0.000000,0.074047
LAMI.RD319.D2.C3,0.005702,0.433072,0.239277,0.095142,0.000000,0.000000,0.226807


In [113]:
# Subset V4_tbl rows where sample is in metaB_tbl
V4_tbl_matched = V4_tbl[V4_tbl.index.isin(metaB_V4_tbl_relative.index)]

top_V4_features = [' g__Staphylococcus', ' g__Lawsonella',
                    ' g__Streptococcus', ' g__Acinetobacter', ' g__Cutibacterium', 'g__Enhydrobacter',
                     ' g__Micrococcus', ' g__Lactobacillus', ' g__Rothia', ' g__Kocuria', ' g__Haemophilus']
# Find the intersection of desired columns and actual DataFrame columns
available_columns = V4_tbl_matched.columns.intersection(top_V4_features)
V4_tbl_matched = V4_tbl_matched[available_columns]

V4_tbl_matched

,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,317.0,131.0,414.0,116.0,475.0,100.0,15.0,1.0,17.0,36.0
LAMI.RD001.D14.C1,454.0,456.0,247.0,57.0,168.0,93.0,117.0,0.0,5.0,15.0
LAMI.RD001.D14.C2,444.0,206.0,457.0,66.0,456.0,54.0,23.0,0.0,5.0,47.0
LAMI.RD017.D14.C2,1756.0,636.0,33.0,70.0,25.0,617.0,2.0,126.0,7.0,1.0
LAMI.RD011.D14.C1,2063.0,999.0,85.0,17.0,37.0,88.0,45.0,32.0,3.0,7.0
...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D25.C2,68.0,54.0,1.0,1.0,0.0,4.0,0.0,1.0,0.0,0.0
LAMI.RD319.D28.C1,42.0,96.0,7.0,5.0,9.0,0.0,0.0,0.0,0.0,2.0
LAMI.RD318.D9.C2,139.0,23.0,11.0,35.0,2.0,12.0,2.0,10.0,0.0,1.0
LAMI.RD319.D28.C2,52.0,54.0,1.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0


In [114]:
# Convert to relative abundance (row-wise normalization)
V4_tbl_matched_relative = V4_tbl_matched.div(V4_tbl_matched.sum(axis=1), axis=0)
V4_tbl_matched_relative

,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Acinetobacter,g__Haemophilus,g__Micrococcus,g__Lactobacillus,g__Cutibacterium,g__Kocuria,g__Rothia
LAMI.RD001.D0.C1,0.195438,0.080764,0.255240,0.071517,0.292848,0.061652,0.009248,0.000617,0.010481,0.022195
LAMI.RD001.D14.C1,0.281638,0.282878,0.153226,0.035360,0.104218,0.057692,0.072581,0.000000,0.003102,0.009305
LAMI.RD001.D14.C2,0.252560,0.117179,0.259954,0.037543,0.259386,0.030717,0.013083,0.000000,0.002844,0.026735
LAMI.RD017.D14.C2,0.536511,0.194317,0.010082,0.021387,0.007638,0.188512,0.000611,0.038497,0.002139,0.000306
LAMI.RD011.D14.C1,0.611078,0.295912,0.025178,0.005036,0.010960,0.026066,0.013329,0.009479,0.000889,0.002073
...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D25.C2,0.527132,0.418605,0.007752,0.007752,0.000000,0.031008,0.000000,0.007752,0.000000,0.000000
LAMI.RD319.D28.C1,0.260870,0.596273,0.043478,0.031056,0.055901,0.000000,0.000000,0.000000,0.000000,0.012422
LAMI.RD318.D9.C2,0.591489,0.097872,0.046809,0.148936,0.008511,0.051064,0.008511,0.042553,0.000000,0.004255
LAMI.RD319.D28.C2,0.468468,0.486486,0.009009,0.000000,0.009009,0.018018,0.009009,0.000000,0.000000,0.000000


In [115]:
# Save metadata for V4
metadata_V4 = metadata[metadata['SampleID'].isin(V4_tbl_matched.index)]
metadata_V4 = metadata_V4[['SampleID', 'group']]
# Rename the first column to 'sample id'
metadata_V4.rename(columns={metadata_V4.columns[0]: 'sample id'}, inplace=True)

# Remove the name of the second column (set it to an empty string)
metadata_V4.columns = ['sample id'] + ['' if i == 1 else col for i, col in enumerate(metadata_V4.columns[1:])]

metadata_V4.to_csv('../Data/multi-omics/16S_V4_metaB-matched.csv', index=0)
metadata_V4

,sample id,group
1,LAMI.RD310.D21.C1,Acne_L
2,LAMI.RD305.D21.C3,Acne_NL
3,LAMI.RD306.D18.C2,Acne_L
6,LAMI.RD317.D14.C2,Acne_L
8,LAMI.RD304.D7.C1,Acne_L
...,...,...
259,LAMI.RD313.D4.C1,Acne_L
260,LAMI.RD305.D0.C2,Acne_L
261,LAMI.RD317.D21.C1,Acne_L
262,LAMI.RD001.D0.C1,Healthy


In [116]:
# Save as biom for mmvec
metaB_V4_tbl_transposed = metaB_V4_tbl_relative.T
output_path = '../Data/multi-omics/metabolomics-tbl_16S_V4-matched_relative.biom'
save_as_biom(metaB_V4_tbl_transposed, output_path)

In [117]:
# Save as biom for mmvec
V4_tbl_matched_transposed = V4_tbl_matched_relative.T
output_path = '../Data/multi-omics/16S_V4-tbl_metaB-matched_relative.biom'
save_as_biom(V4_tbl_matched_transposed, output_path)

### Combine V1-V3 and V4 tables

In [118]:
# # Find common rows (samples)
# common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# # Subset both DataFrames to keep only common rows
# V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
# V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# # Merge by taking average of values row-wise
# merged_tbl = V1V3_tbl_matched.add(V4_tbl_matched, fill_value=0)

# # Sort columns by total sum (descending order)
# merged_tbl = merged_tbl[merged_tbl.sum().sort_values(ascending=False).index]

# merged_tbl

In [119]:
# Find common rows (samples)
common_rows = V1V3_tbl_matched.index.intersection(V4_tbl_matched.index)

# Subset both DataFrames to keep only common rows
V1V3_tbl_matched = V1V3_tbl_matched.loc[common_rows]
V4_tbl_matched = V4_tbl_matched.loc[common_rows]

# Compute the mean for each matching row and column
merged_tbl = (V1V3_tbl_matched + V4_tbl_matched) / 2

# Sort columns by total mean (descending order)
merged_tbl = merged_tbl[merged_tbl.mean().sort_values(ascending=False).index]

# Display merged DataFrame
merged_tbl


,g__Cutibacterium,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Lactobacillus,g__Haemophilus,g__Micrococcus,g__Kocuria,g__Rothia,g__Acinetobacter,g__Alloprevotella,g__Corynebacterium
LAMI.RD001.D0.C1,1181.5,205.5,111.0,353.0,8.0,281.0,68.5,8.5,43.0,NaN,NaN,NaN
LAMI.RD001.D14.C1,1117.0,268.5,354.5,243.5,88.5,88.0,61.5,2.5,13.5,NaN,NaN,NaN
LAMI.RD001.D14.C2,880.5,290.5,178.5,451.5,24.5,257.0,35.0,5.5,39.0,NaN,NaN,NaN
LAMI.RD001.D28.C1,1184.0,349.0,337.0,321.0,50.0,76.0,47.0,5.0,24.0,NaN,NaN,NaN
LAMI.RD004.D0.C1,1340.0,352.5,35.0,290.0,16.0,8.5,52.0,2.5,29.5,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D7.C3,491.0,25.5,4.5,0.0,0.0,0.0,0.0,0.0,0.5,NaN,NaN,NaN
LAMI.RD318.D28.C3,1611.0,38.0,14.0,2.5,1.0,3.0,0.5,0.5,0.0,NaN,NaN,NaN
LAMI.RD319.D14.C1,950.5,18.0,7.0,0.5,0.5,0.0,1.0,0.0,0.0,NaN,NaN,NaN
LAMI.RD319.D14.C2,1157.5,40.0,30.0,1.5,0.5,0.5,0.5,0.0,0.0,NaN,NaN,NaN


In [120]:
# Convert to relative abundance (row-wise normalization)
merged_tbl_relative = merged_tbl.div(merged_tbl.sum(axis=1), axis=0)
merged_tbl_relative

,g__Cutibacterium,g__Staphylococcus,g__Lawsonella,g__Streptococcus,g__Lactobacillus,g__Haemophilus,g__Micrococcus,g__Kocuria,g__Rothia,g__Acinetobacter,g__Alloprevotella,g__Corynebacterium
LAMI.RD001.D0.C1,0.522788,0.090929,0.049115,0.156195,0.003540,0.124336,0.030310,0.003761,0.019027,NaN,NaN,NaN
LAMI.RD001.D14.C1,0.499218,0.120000,0.158436,0.108827,0.039553,0.039330,0.027486,0.001117,0.006034,NaN,NaN,NaN
LAMI.RD001.D14.C2,0.407262,0.134366,0.082562,0.208834,0.011332,0.118871,0.016189,0.002544,0.018039,NaN,NaN,NaN
LAMI.RD001.D28.C1,0.494776,0.145842,0.140827,0.134141,0.020894,0.031759,0.019641,0.002089,0.010029,NaN,NaN,NaN
LAMI.RD004.D0.C1,0.630292,0.165804,0.016463,0.136406,0.007526,0.003998,0.024459,0.001176,0.013876,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
LAMI.RD319.D7.C3,0.941515,0.048897,0.008629,0.000000,0.000000,0.000000,0.000000,0.000000,0.000959,NaN,NaN,NaN
LAMI.RD318.D28.C3,0.964382,0.022748,0.008381,0.001497,0.000599,0.001796,0.000299,0.000299,0.000000,NaN,NaN,NaN
LAMI.RD319.D14.C1,0.972379,0.018414,0.007161,0.000512,0.000512,0.000000,0.001023,0.000000,0.000000,NaN,NaN,NaN
LAMI.RD319.D14.C2,0.940675,0.032507,0.024380,0.001219,0.000406,0.000406,0.000406,0.000000,0.000000,NaN,NaN,NaN


In [121]:
# Save as biom for mmvec
merged_tbl_relative_transposed = merged_tbl_relative.T
output_path = '../Data/multi-omics/16S_MERGED-Avg-tbl_metaB-matched_relative.biom'
save_as_biom(merged_tbl_relative_transposed, output_path)

In [122]:
# Subset metaB_tbl rows where sample is in V1V3_tbl
metaB_MERGED_tbl = metaB_tbl[metaB_tbl.index.isin(merged_tbl_relative.index)]

# Remove the 'group' column (only for VIP table)
# metaB_MERGED_tbl = metaB_MERGED_tbl.drop(columns=['group'])
metaB_MERGED_tbl.index.name = None

In [123]:
# Sort columns by total sum (descending order)
metaB_MERGED_tbl = metaB_MERGED_tbl[metaB_MERGED_tbl.sum().sort_values(ascending=False).index]
metaB_MERGED_tbl

,Isoleucine,Phenylalanine,Tryptophan,C19H36O4 Fatty alcohol,Sorbitane Monooleate,Urocanic acid,N-acyl lipid glutamine-C14:0
LAMI.RD001.D0.C1,2025607.40,970487.500,567510.060,611709.700,0.000,39859.7580,0.0000
LAMI.RD304.D11.C1,1434033.20,595568.940,328151.120,78491.530,99470.690,36750.4000,5375.8525
LAMI.RD304.D11.C2,706197.75,303845.300,152392.720,261996.800,157503.920,0.0000,2691.0256
LAMI.RD304.D0.C1,778566.10,293386.340,137201.390,427202.060,0.000,3345.8184,3431.0034
LAMI.RD304.D0.C2,2486485.20,1051277.400,459814.620,223301.700,62253.426,24984.3220,13703.5760
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,1250164.60,579058.800,498893.780,122958.260,0.000,9787.1210,0.0000
LAMI.RD308.D9.C3,687527.06,281896.700,262886.470,65659.700,0.000,3076.9688,0.0000
LAMI.RD319.D2.C2,538412.94,255542.720,121803.890,73231.560,0.000,0.0000,0.0000
LAMI.RD319.D2.C3,106901.95,59064.336,23485.404,55986.290,0.000,1407.5230,0.0000


In [124]:
# Convert to relative abundance (row-wise normalization)
metaB_MERGED_tbl_relative = metaB_MERGED_tbl.div(metaB_MERGED_tbl.sum(axis=1), axis=0)
metaB_MERGED_tbl_relative

,Isoleucine,Phenylalanine,Tryptophan,C19H36O4 Fatty alcohol,Sorbitane Monooleate,Urocanic acid,N-acyl lipid glutamine-C14:0
LAMI.RD001.D0.C1,0.480551,0.230237,0.134635,0.145121,0.000000,0.009456,0.000000
LAMI.RD304.D11.C1,0.556292,0.231034,0.127297,0.030449,0.038587,0.014256,0.002085
LAMI.RD304.D11.C2,0.445655,0.191746,0.096169,0.165337,0.099395,0.000000,0.001698
LAMI.RD304.D0.C1,0.473830,0.178553,0.083500,0.259992,0.000000,0.002036,0.002088
LAMI.RD304.D0.C2,0.575333,0.243249,0.106394,0.051668,0.014404,0.005781,0.003171
...,...,...,...,...,...,...,...
LAMI.RD318.D9.C3,0.508019,0.235307,0.202731,0.049966,0.000000,0.003977,0.000000
LAMI.RD308.D9.C3,0.528441,0.216669,0.202058,0.050467,0.000000,0.002365,0.000000
LAMI.RD319.D2.C2,0.544406,0.258387,0.123160,0.074047,0.000000,0.000000,0.000000
LAMI.RD319.D2.C3,0.433072,0.239277,0.095142,0.226807,0.000000,0.005702,0.000000


In [125]:
# Save as biom for mmvec
metaB_MERGED_tbl_relative_transposed = metaB_MERGED_tbl_relative.T
output_path = '../Data/multi-omics/metaB_MERGED-Avg-tbl_matched_relative.biom'
save_as_biom(metaB_MERGED_tbl_relative_transposed, output_path)